In [21]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import interpn
import plotly.graph_objects as go
from functions import *


In [ ]:

def spherical_to_cartesian(azimuth_deg, elevation_deg):
    """Конвертируем углы в направляющий вектор"""
    azimuth = np.radians(azimuth_deg)
    elevation = np.radians(elevation_deg)
    x = np.cos(elevation) * np.cos(azimuth)
    y = np.cos(elevation) * np.sin(azimuth)
    z = np.sin(elevation)
    return np.array([x, y, z])

def line_parametric(t, origin, direction):
    """Параметрическое уравнение прямой"""
    return origin + direction * t

def find_intersection(matrix_shape, origin, direction):
    """Находим точки пересечения прямой с границами матрицы"""
    def get_t(axis, boundary):
        return (boundary - origin[axis]) / direction[axis] if direction[axis] != 0 else np.nan
    
    min_bounds = np.array([0, 0, 0])
    max_bounds = np.array(matrix_shape) - 1e-6  # Избегаем краевых эффектов
    
    t_enter = []
    t_exit = []
    
    for i in range(3):
        t1 = get_t(i, min_bounds[i])
        t2 = get_t(i, max_bounds[i])
        t_enter.append(min(t1, t2))
        t_exit.append(max(t1, t2))
    
    t_start = max(0, *[t for t in t_enter if not np.isnan(t)])
    t_end = min(*[t for t in t_exit if not np.isnan(t)])
    
    return t_start, t_end

def trilinear_interpolation(matrix, x, y, z):
    """Трилинейная интерполяция"""
    x0, y0, z0 = np.floor([x, y, z]).astype(int)
    x1, y1, z1 = x0 + 1, y0 + 1, z0 + 1
    
    # Обрабатываем выход за границы
    x0, x1 = np.clip([x0, x1], 0, matrix.shape[0]-1)
    y0, y1 = np.clip([y0, y1], 0, matrix.shape[1]-1)
    z0, z1 = np.clip([z0, z1], 0, matrix.shape[2]-1)
    
    xd = (x - x0) / (x1 - x0) if (x1 - x0) != 0 else 0
    yd = (y - y0) / (y1 - y0) if (y1 - y0) != 0 else 0
    zd = (z - z0) / (z1 - z0) if (z1 - z0) != 0 else 0

    c00 = matrix[x0, y0, z0] * (1 - xd) + matrix[x1, y0, z0] * xd
    c01 = matrix[x0, y0, z1] * (1 - xd) + matrix[x1, y0, z1] * xd
    c10 = matrix[x0, y1, z0] * (1 - xd) + matrix[x1, y1, z0] * xd
    c11 = matrix[x0, y1, z1] * (1 - xd) + matrix[x1, y1, z1] * xd

    c0 = c00 * (1 - yd) + c10 * yd
    c1 = c01 * (1 - yd) + c11 * yd

    return c0 * (1 - zd) + c1 * zd



In [ ]:

# матрица 10x10x10
matrix = np.random.rand(10, 10, 10)
origin = np.array([-2, -2, -2])  # Начальная точка 
azimuth_deg = 45  # Горизонтальный угол
elevation_deg = 0  # Вертикальный угол
num_points = 20  # Количество точек на выходе

# Конвертируем углы в направление
direction = spherical_to_cartesian(azimuth_deg, elevation_deg)

# Находим пересечение с матрицей
t_start, t_end = find_intersection(matrix.shape, origin, direction)

# Генерируем точки вдоль линии
t_values = np.linspace(t_start, t_end, num_points)
points = np.array([line_parametric(t, origin, direction) for t in t_values])

values = [trilinear_interpolation(matrix, x, y, z) for x, y, z in points]

print(points, "\n", np.floor(points).astype(int),"\n", values)

[[ 2.          0.         -2.        ]
 [ 2.42105258  0.42105258 -2.        ]
 [ 2.84210516  0.84210516 -2.        ]
 [ 3.26315774  1.26315774 -2.        ]
 [ 3.68421032  1.68421032 -2.        ]
 [ 4.10526289  2.10526289 -2.        ]
 [ 4.52631547  2.52631547 -2.        ]
 [ 4.94736805  2.94736805 -2.        ]
 [ 5.36842063  3.36842063 -2.        ]
 [ 5.78947321  3.78947321 -2.        ]
 [ 6.21052579  4.21052579 -2.        ]
 [ 6.63157837  4.63157837 -2.        ]
 [ 7.05263095  5.05263095 -2.        ]
 [ 7.47368353  5.47368353 -2.        ]
 [ 7.89473611  5.89473611 -2.        ]
 [ 8.31578868  6.31578868 -2.        ]
 [ 8.73684126  6.73684126 -2.        ]
 [ 9.15789384  7.15789384 -2.        ]
 [ 9.57894642  7.57894642 -2.        ]
 [ 9.999999    7.999999   -2.        ]] 
 [[ 2  0 -2]
 [ 2  0 -2]
 [ 2  0 -2]
 [ 3  1 -2]
 [ 3  1 -2]
 [ 4  2 -2]
 [ 4  2 -2]
 [ 4  2 -2]
 [ 5  3 -2]
 [ 5  3 -2]
 [ 6  4 -2]
 [ 6  4 -2]
 [ 7  5 -2]
 [ 7  5 -2]
 [ 7  5 -2]
 [ 8  6 -2]
 [ 8  6 -2]
 [ 9  7 -2]
 

In [24]:
fig = go.Figure()

x, y, z = np.indices((10, 10, 10))
fig.add_trace(go.Volume(
    x=x.flatten(),
    y=y.flatten(),
    z=z.flatten(),
    opacity=0.2,
    showscale=False
))

fig.add_trace(go.Scatter3d(
    x=points[:, 0],
    y=points[:, 1],
    z=points[:, 2],
    mode='lines+markers',
    line=dict(color=values, colorscale='Viridis'),
    marker=dict(size=3, color=values, colorscale='Viridis'),
    text=[f'{v:.4f}' for v in values],  
    textposition='top center',
    name='Line'
))

fig.update_layout(scene=dict(
    xaxis_title='X',
    yaxis_title='Y',
    zaxis_title='Z',
    aspectmode='cube'
))

fig.show()

In [31]:
print(matrix[2,0,8])
points[0], values[0]

0.9810533202974526


(array([ 2.,  0., -2.]), np.float64(0.32235438255578497))